# 02. Feature Engineering and Selection

## 1. Introduction
This notebook transforms cleaned backlink data into a modeling-ready dataset. Missing quality metrics are imputed using domain-level medians, derived features capture negotiation dynamics and temporal patterns, and categorical variables are encoded for tree-based models. The final feature set is validated through correlation analysis before splitting into train/test partitions.

## 2. Objectives

**O1. Missing value imputation**
Fill CF/TF gaps using domain-level median values, preserving signal where partial observations exist.

**O2. Feature derivation**
Engineer price_ratio (negotiation signal) and temporal features (year, month, quarter) from raw columns.

**O3. Correlation analysis**
Compute and visualize pairwise correlations to validate feature independence and identify multicollinearity.

**O4. Encoding and selection**
Label-encode categorical features (TLD, country), define the final feature set, and create reproducible train/test splits.

## 3. Sections
| # | Section | Purpose |
|---|---------|--------|
| 4 | Environment setup | Imports, logging, paths |
| 5 | Data loading | Load cleaned Parquet from notebook 01 |
| 6 | Missing value imputation | Impute CF/TF using domain-level medians |
| 7 | Feature engineering | Add price_ratio and temporal features |
| 8 | Correlation analysis | Heatmap of numeric feature correlations |
| 9 | Categorical encoding | Label-encode TLD and country |
| 10 | Feature set and train/test split | Define final features, split data |
| 11 | Save engineered datasets | Persist train/test partitions |
| 12 | Summary | Key findings and output artifacts |

In [ ]:
import sys
from pathlib import Path


def _ensure_local_repo_src_on_path() -> None:
    for candidate in (Path.cwd() / "src", Path.cwd().parent / "src"):
        package_root = candidate / "backlink_pricing_model"
        if package_root.exists():
            candidate_str = str(candidate.resolve())
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return


_ensure_local_repo_src_on_path()

In [ ]:
import logging
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from backlink_pricing_model.core.environment import get_project_root
from backlink_pricing_model.core.models.visualization import PlotConfig
from backlink_pricing_model.core.notebook import display_saved_image_or_figure
from backlink_pricing_model.preprocessing.data_imputation import (
    impute_metrics_by_domain,
    summarize_imputation,
)
from backlink_pricing_model.preprocessing.data_loading import save_processed
from backlink_pricing_model.preprocessing.feature_engineering import (
    add_price_ratio,
    add_temporal_features,
)
from backlink_pricing_model.visualization.importance_plots import (
    plot_correlation_heatmap,
)
from backlink_pricing_model.visualization.plots_style import apply_plotly_defaults

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

apply_plotly_defaults()

PROJECT_ROOT = get_project_root()
IMAGE_DIR = PROJECT_ROOT / "images" / "feature_engineering"
ENGINEERED_DATA_DIR = PROJECT_ROOT / "data" / "engineered"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
ENGINEERED_DATA_DIR.mkdir(parents=True, exist_ok=True)

logger.info("Project root: %s", PROJECT_ROOT)

## 5. Data loading

Load the cleaned backlink dataset produced by notebook 01. This dataset has valid prices, normalized categoricals, and log transforms already applied.

In [ ]:
df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "backlinks_cleaned.parquet")
logger.info("Loaded %d rows, %d columns", len(df), len(df.columns))
display(df.head())

## 6. Missing value imputation

CF and TF are missing for ~65.9% of the cleaned dataset. However, many domains appear multiple times in the marketplace with at least one record that has known CF/TF values. Domain-level median imputation exploits this structure: where a domain has partial observations, the median of its known values fills the gaps in its other records. This approach preserves the domain-specific signal that a global median would destroy, while avoiding the bias that more sophisticated imputation methods (e.g., regression-based) might introduce at this stage.

In [ ]:
df_before = df.copy()
df = impute_metrics_by_domain(df)

imputation_summary = summarize_imputation(df_before, df)
logger.info("Imputation summary:")
display(imputation_summary)

Domain-level median imputation recovered 10,696 CF values and 10,697 TF values. This is a substantial recovery — roughly half of the originally missing values now have plausible estimates grounded in same-domain observations. The remaining unimputed rows are domains that appear only once in the dataset with no known CF/TF; these will become NaN rows dropped during the final modeling dataset construction. This trade-off between imputation coverage and data fidelity is intentional: we accept a smaller modeling dataset rather than filling values with global statistics that would dilute per-domain signal.

## 7. Feature engineering

Add derived features that capture negotiation dynamics and temporal pricing patterns:
- **price_ratio**: `final_price / initial_price` — measures negotiation discount
- **year, month, quarter**: temporal features extracted from `date_received`

In [ ]:
df = add_price_ratio(df)
df = add_temporal_features(df)

logger.info("Columns after feature engineering: %s", list(df.columns))
display(df.describe())

## 8. Correlation analysis

Compute pairwise Pearson correlations among numeric features to check for multicollinearity and validate that the target (`log_price`) has meaningful relationships with candidate predictors.

In [ ]:
numeric_cols = [
    "final_price",
    "dr",
    "cf",
    "tf",
    "domain_traffic",
    "log_price",
    "log_traffic",
]
corr_matrix = df[numeric_cols].corr()
fig = plot_correlation_heatmap(
    corr_matrix,
    title="Numeric feature correlations",
    config=PlotConfig(
        save_path=str(IMAGE_DIR / "correlation_heatmap.png"),
    ),
)
display_saved_image_or_figure(IMAGE_DIR / "correlation_heatmap.png", fig)

In [ ]:
The correlation heatmap reveals several important patterns for feature selection. The strongest correlation with `log_price` comes from `final_price` itself (r = 0.86), which is expected since `log_price` is derived from it — this column is excluded from features. Among candidate predictors, `cf` (r = 0.23), `log_traffic` (r = 0.22), and `tf` (r = 0.22) show modest but meaningful linear relationships with the target. Critically, `dr` has a near-zero linear correlation with `log_price` (r = 0.02), confirming the non-linear interaction pattern discovered in notebook 01. The moderate intercorrelation among DR, CF, and TF (r ~ 0.3--0.5) is expected since all three measure aspects of domain authority, but each retains enough independent variance to justify inclusion. `domain_traffic` (r = 0.06) carries minimal linear signal, though its log-transformed version performs better.

_Figure 1. Pearson correlation heatmap of numeric features. DR shows near-zero linear correlation with log_price despite being a known pricing factor — evidence of non-linear interaction effects that tree-based models will capture._

## 9. Categorical encoding

Label-encode TLD (155 classes) and country (48 classes) for tree-based models. Label encoding assigns arbitrary integer IDs without implying ordinal relationships. This is appropriate for decision-tree splits, which evaluate each split point independently and are therefore invariant to the specific integer values assigned. One-hot encoding would create 200+ sparse columns, which is unnecessary for tree-based models and would significantly increase memory usage.

## 9. Categorical encoding

Label-encode TLD and country for tree-based models. These encodings assign integer IDs without implying ordinal relationships, which is appropriate for decision tree splits.

In [ ]:
## 10. Feature set and train/test split

The final feature set consists of 9 predictors: three domain-quality metrics (`dr`, `cf`, `tf`), one traffic measure (`log_traffic`), two categorical encodings (`tld_encoded`, `country_encoded`), and three temporal features (`year`, `month`, `quarter`). Rows with any remaining NaN values in these columns or the target are dropped before splitting, since tree-based models in scikit-learn and XGBoost do not universally handle missing values in all configurations.

## 10. Feature set and train/test split

Define the final modeling feature set and create a reproducible 80/20 train/test split.

After dropping rows with any remaining NaN values, the modeling dataset contains 19,877 rows — a 35% reduction from the 30,778 cleaned rows. This loss is driven primarily by records where CF/TF could not be imputed (single-occurrence domains with no known values) and rows with missing traffic data. The 80/20 split produces 15,901 training samples and 3,976 test samples. While the 35% data loss is significant, the alternative — imputing with global statistics — would inject noise that could inflate apparent model performance without improving real-world accuracy. The retained 19,877 rows represent the subset of the data where all features have defensible values.

In [ ]:
FEATURE_COLS = [
    "dr",
    "cf",
    "tf",
    "log_traffic",
    "tld_encoded",
    "country_encoded",
    "year",
    "month",
    "quarter",
]
TARGET = "log_price"

df_model = df.dropna(subset=[*FEATURE_COLS, TARGET])
logger.info("Modeling dataset: %d rows, %d features", len(df_model), len(FEATURE_COLS))

X = df_model[FEATURE_COLS]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)
logger.info("Train: %d | Test: %d", len(X_train), len(X_test))

---

## 12. Summary

This notebook transformed the 30,778 cleaned backlink rows into a modeling-ready dataset of 19,877 rows with 9 features and a log-price target.

**Key findings:**

- **Imputation recovery:** Domain-level median imputation recovered 10,696 CF values and 10,697 TF values — roughly half of the originally missing observations. The remaining gaps correspond to single-occurrence domains with no known quality metrics.
- **Correlation structure:** Linear correlations with `log_price` are modest across all features: `cf` (0.23), `log_traffic` (0.22), `tf` (0.22), `domain_traffic` (0.06), `dr` (0.02). The near-zero DR correlation is not a bug — it reflects non-linear interaction effects where DR predicts price only in combination with market features (country, TLD). Tree-based models will capture this signal where linear models cannot.
- **Categorical cardinality:** 155 TLD classes and 48 country classes after label encoding. These high-cardinality categoricals are handled via label encoding, which is appropriate for tree-based splits.
- **Data attrition:** NaN removal reduces the dataset from 30,778 to 19,877 rows (35% loss). This is a deliberate trade-off: we prioritize feature-value integrity over sample size, accepting that the model trains on fewer but higher-quality observations.
- **Final split:** Train set of 15,901 rows (80%) and test set of 3,976 rows (20%), stratified by random sampling with a fixed seed for reproducibility.

**Output artifacts:**
- `data/engineered/backlinks_engineered.parquet` — full engineered dataset (30,778 rows with all derived features)
- `data/engineered/train_df.parquet` — training split (15,901 rows)
- `data/engineered/test_df.parquet` — test split (3,976 rows)
- `images/feature_engineering/correlation_heatmap.png` — feature correlation heatmap

In [ ]:
save_processed(df, "backlinks_engineered", subdir="engineered")
logger.info("Saved full engineered dataset (%d rows)", len(df))

train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

save_processed(train_df, "train_df", subdir="engineered")
save_processed(test_df, "test_df", subdir="engineered")
logger.info(
    "Saved train_df (%d rows) and test_df (%d rows) to data/engineered/",
    len(train_df),
    len(test_df),
)

---

## 12. Summary

This notebook transformed the cleaned backlink dataset into a modeling-ready format with imputed metrics, derived features, and encoded categoricals.

**Key findings:**
- **Imputation:** Domain-level median imputation recovered CF/TF values for records where other observations from the same domain had known values.
- **Feature correlations:** DR, CF, and TF are moderately intercorrelated but each carries independent signal. log_traffic provides additional predictive power.
- **Temporal patterns:** Year, month, and quarter features capture market dynamics and seasonal pricing variation.
- **Final feature set:** 9 features (dr, cf, tf, log_traffic, tld_encoded, country_encoded, year, month, quarter) predicting log_price.

**Output artifacts:**
- `data/engineered/backlinks_engineered.parquet` — full engineered dataset
- `data/engineered/train_df.parquet` — training split (80%)
- `data/engineered/test_df.parquet` — test split (20%)
- `images/feature_engineering/correlation_heatmap.png` — feature correlation heatmap